# 04 — Inference Demo

**AutoClaim AI** | IE University Deep Learning Final Project

End-to-end inference pipeline: image → damage class → triage decision.

**Triage logic** (thresholds in `config.py`):

| Prediction | Confidence | Decision |
|------------|------------|----------|
| dent / scratch | ≥ 80% | 🟢 Fast-Track Claim |
| dent / scratch | < 80% | 🟡 Human Review Required |
| severe damage | ≥ 70% | 🔴 Priority Assessment |
| severe damage | < 70% | 🟡 Human Review Required |
| unclear / bad image | any | 🟡 Human Review Required |

The best available model is loaded automatically (`load_model()` prefers the transfer model).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config, json
from src.predict import load_model, predict_image
from src.triage import get_triage_decision

## 1. Load Model

In [ ]:
model = load_model()  # auto-selects transfer model if available, else custom CNN
print("Model:", model.name)
print("Input shape:", model.input_shape)
print("Output classes:", model.output_shape[-1])

## 2. Sample Test Images

In [ ]:
import random
from pathlib import Path

test_images = list((Path(config.PROC_DIR) / "test").rglob("*.jpg"))
random.seed(42)
sample = random.sample(test_images, min(9, len(test_images)))
print(f"Selected {len(sample)} test images")

## 3. Predict and Triage Grid

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, img_path in zip(axes.flat, sample):
    true_class = img_path.parent.name.replace("_", " ")
    result = predict_image(model, img_path)
    triage = get_triage_decision(result["predicted_class"], result["confidence"])
    img = Image.open(img_path)
    ax.imshow(img)
    correct = true_class == result["predicted_class"]
    ax.set_title(
        f"True: {true_class}\nPred: {result['predicted_class']} ({result['confidence']:.0%})\n{triage['icon']} {triage['decision']}",
        fontsize=8, color="green" if correct else "red"
    )
    ax.axis("off")
plt.suptitle("Inference Demo — Green = Correct, Red = Wrong", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(config.FIG_DIR, "inference_demo.png"), dpi=150)
plt.show()

## 4. Full JSON Output

In [ ]:
test_img = sample[0]
result = predict_image(model, test_img)
triage = get_triage_decision(result["predicted_class"], result["confidence"])

print(json.dumps({
    "image":           test_img.name,
    "true_class":      test_img.parent.name.replace("_", " "),
    "predicted_class": result["predicted_class"],
    "confidence":      result["confidence"],
    "top_3":           result["top_3"],
    "triage":          triage,
}, indent=2))

## 5. Preprocessing Pipeline Visualised

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage

img_path = sample[1]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

original = PILImage.open(img_path).convert("RGB")
axes[0].imshow(original)
axes[0].set_title(f"1. Original\n{original.width}×{original.height} px")
axes[0].axis("off")

resized = original.resize(config.IMG_SIZE)
axes[1].imshow(resized)
axes[1].set_title(f"2. Resized to {config.IMG_SIZE[0]}×{config.IMG_SIZE[1]}")
axes[1].axis("off")

# The model receives [0,255] float32; Rescaling(1/255) is the first model layer
arr_255 = np.array(resized, dtype=np.float32)  # what the model actually receives
norm = arr_255 / 255.0                          # display-only normalisation
axes[2].imshow(norm)
axes[2].set_title(f"3. After model Rescaling(1/255)\nmean={norm.mean():.3f}")
axes[2].axis("off")

plt.suptitle("Preprocessing Pipeline (step 3 happens inside the model)")
plt.tight_layout()
plt.show()

## 6. Batch Triage Simulation

Simulates the production use case: a batch of uploaded images → triage table.

In [ ]:
import pandas as pd

rows = []
for img_path in sample:
    true_class = img_path.parent.name.replace("_", " ")
    result = predict_image(model, img_path)
    triage = get_triage_decision(result["predicted_class"], result["confidence"])
    rows.append({
        "image":     img_path.name,
        "true":      true_class,
        "predicted": result["predicted_class"],
        "conf":      f"{result['confidence']:.0%}",
        "triage":    f"{triage['icon']} {triage['decision']}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))